<a href="https://colab.research.google.com/github/VENOMDANGER2004/Projects/blob/main/PROJECTSKECH2IMG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install diffusers transformers accelerate opencv-python -q

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image, ImageOps
from diffusers import StableDiffusionXLAdapterPipeline, T2IAdapter, AutoencoderKL, EulerAncestralDiscreteScheduler

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


# **SETUP**

In [ ]:
try:
    # 1. Load T2I Adapter (The 150MB "Tiny ControlNet")
    # We use 'sketch' which is specifically designed for hand drawings
    adapter = T2IAdapter.from_pretrained(
        "TencentARC/t2i-adapter-sketch-sdxl-1.0",
        torch_dtype=torch.float16,
        adapter_type="full_adapter_xl"
    )

    # 2. Load VAE (Fixed 16-bit VAE for better colors)
    vae = AutoencoderKL.from_pretrained(
        "madebyollin/sdxl-vae-fp16-fix",
        torch_dtype=torch.float16,
        use_safetensors=True
    )

    # 3. Load Juggernaut XL v9 (Top-Tier Realism)
    pipe = StableDiffusionXLAdapterPipeline.from_pretrained(
        "segmind/SSD-1B",
        adapter=adapter,
        vae=vae,
        torch_dtype=torch.float16,
        use_safetensors=True
    )

    # 4. Use "Euler A" Scheduler (Fast & High Quality for Juggernaut)
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

    # 5. MAX SURVIVAL MODE (Essential for T4)
    pipe.enable_model_cpu_offload() # Parks the heavy AI in RAM when not used
    pipe.enable_vae_slicing()       # Chops image to prevent VRAM spikes
    pipe.enable_vae_tiling()        # "Infinite" Resolution protection

except Exception as e:
    print(f"❌ Error: {e}")

model_index.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--segmind--SSD-1B/snapshots/60987f37e94cd59c36b1cba832b9f97b57395a10/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: /root/.cache/huggingface/hub/models--segmind--SSD-1B/snapshots/60987f37e94cd59c36b1cba832b9f97b57395a10/text_encoder_2
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ System Ready! (Running Juggernaut XL on 'Survival Mode')


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionXLAdapterPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(
/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2213: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionXLAdapterPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(


# IMAGE **PROCESSING**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image, ImageOps
import io
import numpy as np
import cv2

In [ ]:
STYLE_CONFIG = {
    "Cyberpunk / Sci-Fi": {
        "positive": "cyberpunk city, neon lights, night, rain, reflection, futuristic, unreal engine 5, volumetric fog, detailed texture, 8k, masterpiece",
        "negative": "daylight, sun, rustic, vintage, low contrast, blur, cartoon, drawing,melting"
    },
    "Wong Kar-wai (Cinematic)": {
        "positive": "cinematic film still, moody lighting, hong kong street at night, Background neon sign, shot on 35mm, bokeh, film grain, emotional, masterpiece, vignette",
        "negative": "cartoon, 3d render, bright, flat lighting, clean, sterile, drawing, anime"
    },
    "Anime / Studio Ghibli": {
        "positive": "anime style, studio ghibli, vibrant colors, cel shaded, highly detailed, beautiful scenery, 2d animation aesthetic, makoto shinkai",
        "negative": "photorealistic, 3d, realistic, photograph, sketch, rough, live action"
    },
    "Realistic / Architecture": {
        "positive": "raw photo, dslr, 8k, f/8, sharp focus, high fidelity, intricate details, professional photography, hyperrealistic, award winning",
        "negative": "painting, drawing, illustration, cartoon, anime, blur, bokeh, sketch"
    }
}


In [ ]:
def process_sketch(input_image, user_prompt, user_negative, guidance_scale, control_strength, style_key):
    # 1. RESIZE (1024px Native for Juggernaut)
    w, h = input_image.size
    target_size = 1024
    ratio = min(target_size / w, target_size / h)
    new_w = int(w * ratio)
    new_h = int(h * ratio)

    # Must be divisible by 32 for SDXL
    new_w = new_w - (new_w % 32)
    new_h = new_h - (new_h % 32)

    input_resized = input_image.resize((new_w, new_h), Image.Resampling.LANCZOS)
    image_array = np.array(input_resized)

    # 2. ADAPTER PRE-PROCESSOR (Edge Detection)
    # T2I-Sketch wants a clean "Edge Map".
    if len(image_array.shape) == 3:
        grayscale = cv2.cvtColor(image_array, cv2.COLOR_RGB2GRAY)
    else:
        grayscale = image_array

    # Use Adaptive Thresholding to find lines in your sketch
    # This works better than Canny for hand drawings
    inverted = 255 - grayscale
    detected_map = cv2.adaptiveThreshold(
        inverted, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )

    # Clean up noise
    kernel = np.ones((2,2), np.uint8)
    detected_map = cv2.morphologyEx(detected_map, cv2.MORPH_OPEN, kernel)

    detected_map = detected_map[:, :, None]
    detected_map = np.concatenate([detected_map, detected_map, detected_map], axis=2)
    control_image = Image.fromarray(detected_map)

    # 3. PROMPT
    config = STYLE_CONFIG.get(style_key, {"positive": "", "negative": ""})
    final_pos = f"{user_prompt}, {config['positive']}"
    final_neg = f"{user_negative}, {config['negative']}, low quality, watermark, blurry, distorted, ugly"

    # 4. RUN
    output = pipe(
        final_pos,
        image=control_image,
        negative_prompt=final_neg,
        num_inference_steps=45, # Juggernaut likes 30-40
        guidance_scale=guidance_scale, # 5.0 - 7.0 is best
        adapter_conditioning_scale=control_strength, # T2I Adapter Strength
        width=new_w,
        height=new_h
    ).images[0]

    return control_image, output

# UI **INTERFACE**

In [ ]:
# ==========================================
# CELL 3: UI INTERFACE
# ==========================================
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io

header = widgets.HTML("<h2>🎬 The Director's Canvas</h2>")
upload_btn = widgets.FileUpload(accept='image/*', multiple=False, description='Upload Sketch')
prompt_box = widgets.Text(description='Prompt:', placeholder='Describe scene...', value='A cinematic shot of a duck in Shibuya', layout=widgets.Layout(width='95%'))
style_dropdown = widgets.Dropdown(options=["Cyberpunk / Sci-Fi", "Wong Kar-wai (Cinematic)", "Anime / Studio Ghibli", "Realistic / Architecture"], value="Cyberpunk / Sci-Fi", description='Style:')

# Settings (Standard SD 1.5 Values)
guidance_slider = widgets.FloatSlider(value=7.5, min=1.0, max=15.0, step=0.5, description='Prompt Force:')
strength_slider = widgets.FloatSlider(value=0.7, min=0.0, max=2.0, step=0.1, description='Sketch Force:')
neg_box = widgets.Text(description='Negative:', placeholder='Optional...', value='', layout=widgets.Layout(width='95%'))

advanced = widgets.Accordion(children=[widgets.VBox([widgets.Label("🔧 Settings"), guidance_slider, strength_slider, neg_box])])
advanced.set_title(0, '⚙️ Advanced Settings')
advanced.selected_index = None

run_btn = widgets.Button(description='✨ Generate', button_style='primary', layout=widgets.Layout(width='100%'))
out = widgets.Output()

def on_button_click(b):
    with out:
        clear_output()
        if not upload_btn.value:
            print("⚠️ Please upload a sketch first!")
            return

        print(f"🚀 Processing..")
        try:
            uploaded_file = upload_btn.value[0] if isinstance(upload_btn.value, tuple) else list(upload_btn.value.values())[0]
            input_pil = Image.open(io.BytesIO(uploaded_file['content'])).convert("RGB")

            control, result = process_sketch(
                input_pil, prompt_box.value, neg_box.value,
                guidance_slider.value, strength_slider.value, style_dropdown.value
            )

            def img_to_bytes(img):
                b = io.BytesIO(); img.save(b, format='PNG'); return b.getvalue()

            display(widgets.HBox([
                widgets.Image(value=img_to_bytes(control), format='png', width=250),
                widgets.Image(value=img_to_bytes(result), format='png', width=250)
            ]))
            display(widgets.Image(value=img_to_bytes(result), format='png', width=600))
        except Exception as e:
            print(f"❌ Error: {e}")

run_btn.on_click(on_button_click)
display(header, upload_btn, prompt_box, style_dropdown, advanced, run_btn, out)

HTML(value="<h2>🎬 The Director's Canvas</h2>")

FileUpload(value={}, accept='image/*', description='Upload Sketch')

Text(value='A cinematic shot of a duck in Shibuya', description='Prompt:', layout=Layout(width='95%'), placeho…

Dropdown(description='Style:', options=('Cyberpunk / Sci-Fi', 'Wong Kar-wai (Cinematic)', 'Anime / Studio Ghib…

Accordion(children=(VBox(children=(Label(value='🔧 Settings'), FloatSlider(value=7.5, description='Prompt Force…

Button(button_style='primary', description='✨ Generate', layout=Layout(width='100%'), style=ButtonStyle())

Output()